# Installation
SAM2 installation and dependencies.

Paths containing `Ablation_test` below retain the legacy Google Drive folder name used during development; they do not refer to an active Git branch.


In [ ]:
# Import core Python and CUDA-related packages for the Colab runtime.
## Dependencies
import torch
import sys
import torchvision
import cuda

In [ ]:
# Clone the SAM2 repository and install it in editable mode.
!git clone https://github.com/facebookresearch/sam2.git
%cd sam2
!pip install -e.

In [ ]:
# Download the SAM2 checkpoints used by the mask generator.
!cd checkpoints && ./download_ckpts.sh && cd ..

# Segmentation
Segmentation workflow overview.


## Set-up
Runtime setup for SAM2 segmentation.


In [ ]:
# Flag that the notebook is running in a Colab environment.
using_colab = True

In [ ]:
# Install additional Colab dependencies and download a default SAM2 checkpoint.
if using_colab:
    import torch
    import torchvision
    print("PyTorch version:", torch.__version__)
    print("Torchvision version:", torchvision.__version__)
    print("CUDA is available:", torch.cuda.is_available())
    import sys
    !{sys.executable} -m pip install opencv-python matplotlib
    !{sys.executable} -m pip install 'git+https://github.com/facebookresearch/sam2.git'

    !mkdir -p images
    !wget -P images https://raw.githubusercontent.com/facebookresearch/sam2/main/notebooks/images/cars.jpg

    !mkdir -p ../checkpoints/
    !wget -P ../checkpoints/ https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_large.pt

In [ ]:
# Import common libraries and configure PyTorch fallback behavior.
import os
# if using Apple MPS, fall back to CPU for unsupported ops
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"
import numpy as np
import torch
import matplotlib.pyplot as plt
from PIL import Image

In [ ]:
# Select the computation device and enable CUDA precision optimizations when available.
# select the device for computation
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"using device: {device}")

if device.type == "cuda":
    # use bfloat16 for the entire notebook
    torch.autocast("cuda", dtype=torch.bfloat16).__enter__()
    # turn on tfloat32 for Ampere GPUs (https://pytorch.org/docs/stable/notes/cuda.html#tensorfloat-32-tf32-on-ampere-devices)
    if torch.cuda.get_device_properties(0).major >= 8:
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
elif device.type == "mps":
    print(
        "\nSupport for MPS devices is preliminary. SAM 2 is trained with CUDA and might "
        "give numerically different outputs and sometimes degraded performance on MPS. "
        "See e.g. https://github.com/pytorch/pytorch/issues/84936 for a discussion."
    )

In [ ]:
# Define a helper to visualize SAM2 mask annotations.
import numpy as np
import matplotlib.pyplot as plt
import cv2
np.random.seed(3)

def show_anns(anns, borders=True):
    if len(anns) == 0:
        return
    sorted_anns = sorted(anns, key=(lambda x: x['area']), reverse=True)
    ax = plt.gca()
    ax.set_autoscale_on(False)

    img = np.ones((sorted_anns[0]['segmentation'].shape[0], sorted_anns[0]['segmentation'].shape[1], 4))
    img[:, :, 3] = 0
    for ann in sorted_anns:
        m = ann['segmentation']
        color_mask = np.concatenate([np.random.random(3), [0.5]])
        img[m] = color_mask
        if borders:
            import cv2
            contours, _ = cv2.findContours(m.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
            # Try to smooth contours
            contours = [cv2.approxPolyDP(contour, epsilon=0.01, closed=True) for contour in contours]
            cv2.drawContours(img, contours, -1, (0, 0, 1, 0.4), thickness=1)

    ax.imshow(img)

## Mask generator
SAM2 automatic mask generation.


In [ ]:
# Mount Google Drive for reading images and writing generated masks.
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
# Build the SAM2 model and automatic mask generator.
%cd sam2
from sam2.build_sam import build_sam2
from sam2.automatic_mask_generator import SAM2AutomaticMaskGenerator

sam2_checkpoint = "../checkpoints/sam2.1_hiera_base_plus.pt"
model_cfg = "configs/sam2.1/sam2.1_hiera_b+.yaml"

sam2 = build_sam2(model_cfg, sam2_checkpoint, device=device, apply_postprocessing=False)

mask_generator = SAM2AutomaticMaskGenerator(sam2)

In [ ]:
# Generate one binary mask per input image using the largest SAM2 mask.
import os
import numpy as np
from PIL import Image
from tqdm import tqdm

# cartelle
#input_dir = "/content/drive/MyDrive/Tesi/neus/Ablation_test/Datasets/Sat_simple/Light_simple/Mask_GT/Pose_GT/CORTO_neus/image
input_dir = "/content/drive/MyDrive/Tesi/neus/Ablation_test/Datasets/Sat_simple/Light_realistic/Mask_Segmentation/Pose_COLMAP/Second_Orbit_80/CORTO_src/img"
output_dir = "/content/drive/MyDrive/Tesi/neus/Ablation_test/Datasets/Sat_simple/Light_realistic/Mask_Segmentation/Pose_COLMAP/Second_Orbit_80/CORTO_src/mask"

os.makedirs(output_dir, exist_ok=True)

images = sorted([f for f in os.listdir(input_dir) if f.endswith(".png")])

for img_name in tqdm(images):

    img_path = os.path.join(input_dir, img_name)

    image = Image.open(img_path)
    image = np.array(image.convert("RGB"))

    masks = mask_generator.generate(image)

    if len(masks) == 0:
        print("No mask found:", img_name)
        continue

    largest_mask = max(masks, key=lambda x: x["area"])
    mask = largest_mask["segmentation"]

    binary_mask = np.where(mask, 0, 255).astype(np.uint8)

    out_path = os.path.join(output_dir, img_name)
    Image.fromarray(binary_mask).save(out_path)

print("Masks generation completed")

# NeuS Dataset
NeuS dataset-linking step.


In [ ]:
# Create or replace a symlink so NeuS can reuse the selected image folder.
## Symlink images
import os
import shutil
from pathlib import Path

# ==============================
# INPUT
# ==============================
source_path = "/content/drive/MyDrive/Tesi/neus/Ablation_test/Datasets/Sat_simple/Light_realistic/Mask_GT/Pose_GT/CORTO_neus/image"   # cartella reale
link_path   = "/content/drive/MyDrive/Tesi/neus/Ablation_test/Datasets/Sat_simple/Light_realistic/Mask_Segmentation/Pose_GT/CORTO_neus/image"      # symlink da creare

source_path = Path(source_path)
link_path = Path(link_path)

# ==============================
# CHECK SORGENTE
# ==============================
if not source_path.exists():
    raise FileNotFoundError(f"La cartella sorgente non esiste: {source_path}")

if not source_path.is_dir():
    raise ValueError(f"La sorgente non è una cartella: {source_path}")

# ==============================
# SE ESISTE GIÀ, LO RIMUOVE
# ==============================
if link_path.is_symlink():
    link_path.unlink()
    print(f"Symlink esistente rimosso: {link_path}")

elif link_path.exists():
    if link_path.is_dir():
        shutil.rmtree(link_path)
        print(f"Cartella esistente rimossa: {link_path}")
    else:
        link_path.unlink()
        print(f"File esistente rimosso: {link_path}")

# ==============================
# CREA SYMLINK
# ==============================
os.symlink(source_path, link_path)

print("\nSymlink creato con successo:")
print(f"{link_path} -> {source_path}")